"Rooted trees (...) 'encode' the nature of hierarchy". They abide by a trifecta:
* G is connected
* G does not contain a cycle
* G has n-1 edges
**where any two statements imply the third**
(pg. 78)

In [195]:
from dataclasses import dataclass


e = [(2,1),(2,3),(4,2),(4,6),(6,5),(6,7)]
n = [1,2,3,4,5,6,7]

@dataclass
class Graph:
    edges: list[tuple[int, int]]
    nodes: list[int]
    V: int = 0
    E: int = 0

    def __post_init__(self):
        V = len(self.nodes)
        E = len(self.edges)

my_tree = Graph(edges=e, nodes=n)

## Graph connectivity and graph traversal
"Suppose we are given a graph G = (V, E) and two particular nodes s and t. We'd like (...) an efficient algorithm that answers (...): is there a path from s to t in G?" (pg. 78)

"(...) the s-t Connectivity Problem could also be called the Maze-Solving Problem." (pg. 78)

## Breadth-first Search (BFS)
"Perhaps the simplest algorithm for determining s-t connectivity (...) we explore outward from s in all possible directions, adding nodes one “layer” at a time." (pg. 79)

"... there is a natural physical interpretation to the
algorithm. Essentially, we start at s and “flood” the graph with an **expanding
wave** that grows to visit all nodes that it can reach." (pg. 79)



In [196]:
import random, copy
from collections import deque


# starting at a given node, go through all edges to find new nodes.
# repeat for every found node. O(nm). Expensive.
def bfs(G: Graph, root_node_index: int) -> list[int]: 
    discovered = set()
    q = deque()
    root_node = G.nodes[root_node_index]

    nodes_found = list[int]()
    # edges = copy.deepcopy(G.edges)
    q.append(root_node)
    discovered.add(q[0])
    while len(q) > 0:
        current_node = q[0]
        for edge in G.edges:
            if edge[0] == current_node and edge[1] not in discovered:
                discovered.add(edge[1])
                q.append(edge[1])
            elif edge[1] == current_node and edge[0] not in discovered:
                discovered.add(edge[0])
                q.append(edge[0])
        nodes_found.append(current_node)
        q.popleft()
    return nodes_found

assert(bfs(my_tree,5)==[6,4,5,7,2,1,3])
assert(bfs(my_tree,6)==[7,6,4,5,2,1,3])


my_tree_bfs = Graph(edges=[(1,2),(1,3), (2,4), (2,5), (3,6), (3,7), (4,8), (4,9), (5,10), 
                       (5,11),(6,12),(6,13),(7,14), (7,15)],
                nodes=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15])

rand_root_node_index = random.randrange(0,len(my_tree_bfs.nodes))

print("Root node: ", my_tree_bfs.nodes[rand_root_node_index])
print("  BFS: ", bfs(my_tree_bfs, rand_root_node_index))
print("Root node: 1")
print("  BFS: ", bfs(my_tree_bfs, 0))



Root node:  15
  BFS:  [15, 7, 3, 14, 1, 6, 2, 12, 13, 4, 5, 8, 9, 10, 11]
Root node: 1
  BFS:  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


"Layer L<sub>1</sub> consists of all nodes that are neighbors of *s*" (pg. 80) where *s* is some starting node (in essence, the "new" root). 

"Assuming that we have defined layers L<sub>1</sub>, ..., L<sub>j</sub>, then layer L<sub>j+1</sub> consists of all nodes that do not belong to an earlier layer and that have an edge to a node in layer L<sub>j</sub>." (pg. 80)

L<sub>j</sub> is the set of all nodes at distance *j* from *s*. A node that does not appear in BFS's output fis a node with no path to it. It follows then that BFS does not only show what nodes are reachable starting at s, but also the shortest paths to them. (pg. 80).

"**(3.3)** For each J >= 1, layer L<sub>j</sub> produced by BFS consists of all nodes at distance exactly *j* from *s*. There is a path from *s* to *t* if and only if *t* appears in some layer." (pg. 80)

"The set of nodes discovered by the BFS algorithm is precisely those reachable from the starting node *s*. We will refere to this set *R* as the *connected component* of *G* containing *s*" (pg. 82)

In [197]:
# We can use some helper classes to build our adjecency list...
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None
class LinkedList:
    def __init__(self):
        self.head = None  # The entry point of the list
    # Add a node at the end of the list
    def append(self, data):
        new_node = Node(data)
        if not self.head:
            self.head = new_node
            return
        current = self.head
        while current.next:  # Traverse to the last node
            current = current.next
        current.next = new_node

# both creating the adjancency list and walking it for a BFS tree take O(m+n).
# the exact details in the book aside, the key is that for every node visited _u_, 
# its incident n_u edges are checked. The sum of incident edges is 2m, so the total
# time checking all incident edges is O(m). In a connected graph, n nodes are inspected
# in our outer loop. So the time complexity is O(m+n)
def bfs2(G: Graph, starting_node: int) -> list[int]:
    adj_list = dict[int, LinkedList]((node, LinkedList()) for node in G.nodes)

    # Now we fill the adjencency list
    for edge in G.edges:
        pair = (0,1)
        for i in range(2):
            if edge[pair[0]] in adj_list:
                list_head = adj_list[edge[pair[0]]].head
                while(list_head is not None and list_head.data is not edge[pair[1]]):
                    list_head = list_head.next
                if list_head is None:
                    adj_list[edge[pair[0]]].append(edge[pair[1]])
            pair = (1, 0)

    discovered = list()
    q = deque[int]()
    v = set[int]()
    q.append(starting_node)
    v.add(q[0])
    while len(q) > 0:
        c_node = q[0]
        list_head = adj_list[c_node].head
        while list_head is not None:
            if (list_head.data not in v):
                v.add(list_head.data)
                q.append(list_head.data)
            list_head = list_head.next
        discovered.append(c_node)
        q.popleft()
    return discovered

assert(set(bfs2(my_tree,5))==set([6,4,5,7,2,1,3]))
assert(set(bfs2(my_tree_bfs, 6))==set([7, 3, 14, 15, 1, 6, 2, 12, 13, 4, 5, 8, 9, 10, 11]))



In [ ]:
import time
import pandas as pd

my_tree_bfs = Graph(edges=[(1,2),(1,3), (2,4), (2,5), (3,6), (3,7), (4,8), (4,9), (5,10), 
                       (5,11),(6,12),(6,13),(7,14), (7,15),(8,16),(8,17),(9,18),(9,19),
                       (10,20),(10,21),(11,22),(11,23),(12,24),(12,25),(13,26),(13,27),
                       (14,28),(14,29),(15,30),(15,31),
                       (1,15),(2,21),(3,29),(3,27),(4,11),(4,29),(4,14),(5,6),(5,31),(5,30),
                       (5,19),(5,1),(6,10),(6,31),(6,30),(6,27),(7,1),(7,11)],
                nodes=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,
                       26,27,28,29,30,31])

# For these m > n, so O(m + n) = O(m), and O(m*n)=O(m^2)
print("Running first bfs(): ")


root_indices = [random.randrange(1, len(my_tree_bfs.nodes)) for _ in range(200)]
start = time.perf_counter()
for i in root_indices:
    bfs(my_tree_bfs,i)
end = time.perf_counter()
first_time = f"{end - start:.6f}"
print(f"    Total: {first_time} seconds")

print("Running second bfs(): ")
start = time.perf_counter()
for i in root_indices:
    bfs2(my_tree_bfs,i)
end = time.perf_counter()
second_time = f"{end - start:.6f}"
print(f"    Total: {second_time} seconds")

if float(second_time) >= float(first_time):
    print(pd.Series(root_indices).value_counts())

Running first bfs(): 
    Total: 0.023971 seconds
Running second bfs(): 
    Total: 0.018084 seconds
